# LAI HMM v4 Wrapper Example

This notebook is now a thin driver around `run_lai_hmm(...)` in `LAI_HMM_v4.py`. Edit the file paths and parameters below, then run the wrapper cell.

Use one genotype input or both:

- VCF only: set `vcf_path` and `variant_profiles_path`; leave hap-genotype paths as `None`.
- hap_genotype only: set `hap_genotype_path` and `hap_freq_lookup_path`; leave VCF paths as `None`.
- Combined: set both VCF and hap_genotype inputs. When both inputs are available for a sample, emissions are combined.

See `LAI_HMM_v4_README.md` for Python setup, input format details, and CLI examples.


In [ ]:
from LAI_HMM_v4 import run_lai_hmm


In [ ]:
# -----------------------------
# 1) Set input and output paths
# -----------------------------

# Required for all runs unless marker positions are inferred from a VCF.
marker_positions_path = "marker_positions.csv"
outdir = "results"

# VCF input. Set both to None for hap_genotype-only runs.
vcf_path = None  # e.g. "example.vcf.gz"
variant_profiles_path = None  # e.g. "clade_variant_profiles.tsv"

# hap_genotype input. Set these to None for VCF-only runs.
hap_genotype_path = None  # e.g. "hap_genotype.tsv.gz"
hap_freq_lookup_path = None  # e.g. "allele_freq_lookup.pkl"

# Required only when combining VCF + hap_genotype, because the mixer uses informativeness scores.
hap_informativeness_path = None  # e.g. "haplotype_alleleID_informativeness_scores.txt"

# Optional reference/plotting inputs.
strict_alleles_path = None
mus_hap_alleles_path = None
nonmus_hap_alleles_path = None
chrom_lengths_path = None


In [ ]:
# -----------------------------
# 2) Select samples and parameters
# -----------------------------

# Use sample="SAMPLE1" for one sample, samples=[...] for several, or all_samples=True.
sample = None
samples = None
all_samples = False

# For combined files with all_samples=True, "auto" runs samples present in both inputs.
# Use "union" to run samples present in either input.
sample_source = "auto"

threads = 1
clades = ["EA", "Mus", "NA1", "NA2", "Vv"]

# HMM tuning parameters. Defaults are shown here so runs are explicit.
hmm_params = dict(
    lam_per_Mb=0.05,
    strict_boost=5.0,
    kappa=0.0,
    trans_temp=1.0,
    alpha=0.4,
    windowsize=6,
    e_geno=0.01,
    e_homo=0.1,
    b0=0.0,
    certainty_softener=3.0,
    hom_soften_delta=0.06,
    hom_soften_width=0.08,
    hom_min_mix=0.70,
    hom_neutral=0.60,
    tau=0.0,
)


In [ ]:
# -----------------------------
# 3) Run the wrapper
# -----------------------------

if vcf_path is None and hap_genotype_path is None:
    raise ValueError("Set vcf_path, hap_genotype_path, or both before running this cell.")

run_result = run_lai_hmm(
    vcf_path=vcf_path,
    hap_genotype_path=hap_genotype_path,
    marker_positions_path=marker_positions_path,
    variant_profiles_path=variant_profiles_path,
    hap_freq_lookup_path=hap_freq_lookup_path,
    strict_alleles_path=strict_alleles_path,
    hap_informativeness_path=hap_informativeness_path,
    mus_hap_alleles_path=mus_hap_alleles_path,
    nonmus_hap_alleles_path=nonmus_hap_alleles_path,
    chrom_lengths_path=chrom_lengths_path,
    outdir=outdir,
    sample=sample,
    samples=samples,
    all_samples=all_samples,
    sample_source=sample_source,
    threads=threads,
    clades=clades,
    make_plots=chrom_lengths_path is not None,
    verbose=True,
    **hmm_params,
)

run_result["summary_df"].head()


## Command-Line Equivalent

Run the same workflow without opening the notebook:

```bash
python LAI_HMM_v4.py \
  --vcf cohort.vcf.gz \
  --hap-genotype hap_genotype.tsv.gz \
  --variant-profiles clade_variant_profiles.tsv \
  --hap-frequencies allele_freq_lookup.pkl \
  --hap-informativeness haplotype_alleleID_informativeness_scores.txt \
  --marker-positions marker_positions.csv \
  --all-samples \
  --threads 4 \
  --outdir results
```

For VCF-only runs, omit `--hap-genotype`, `--hap-frequencies`, and `--hap-informativeness`. For hap_genotype-only runs, omit `--vcf` and `--variant-profiles`.
